# Pipeline — full combined run

Bootstraps every agent via `%run` (common, research_workflow, manager, brainstorm, draft,
synthesizer) and runs utterance -> manager -> parallel brainstorm->draft fan-out (`run_prep`)
-> synthesize (`run_full`). Sets `RUN_DEMO = False` before loading so no dependency demo fires.

In [ ]:
import os
# Run from backend/ so `%run` and .env resolve regardless of the kernel's launch dir.
if not os.path.exists("common.ipynb"):
    for _p in ("backend", "extras/socialApp/backend"):
        if os.path.exists(f"{_p}/common.ipynb"):
            os.chdir(_p)
            break
_PIPE_WAS_TOP = globals().get("RUN_DEMO", True)    # unique names: dependencies must not clobber these
_PIPE_WAS_LIVE = globals().get("RUN_LIVE", False)
RUN_DEMO = False                                    # suppress every dependency's demo during %run
RUN_LIVE = False                                    # suppress resolver live probe during %run
%run common.ipynb
%run research_workflow.ipynb
%run manager.ipynb
%run hybrid_resolver.ipynb
%run brainstorm.ipynb
%run draft.ipynb
%run synthesizer.ipynb
RUN_DEMO = _PIPE_WAS_TOP
RUN_LIVE = _PIPE_WAS_LIVE


In [8]:
async def run_planner_for_subject(
    subject: str,
    phrase: str,
    relationship: str,
    max_research_calls: int = MAX_RESEARCH_CALLS_PER_PLANNER,
):
    """Stage 1 (brainstorm) -> stage 2 (draft) -> guards. Returns (bs, out, n_research)."""
    base = (
        f"Subject: {subject}\n"
        f"Phrase: {phrase}\n"
        f"Relationship: {relationship}"
    )

    bs_result = await Runner.run(brainstorm_agent, base, max_turns=2)
    bs: BrainstormOutput = bs_result.final_output

    draft_input = (
        base + "\n"
        f"Research budget: you can and should call the `research` tool up to "
        f"{max_research_calls} times for this subject.\n"
        + render_brainstorm(bs)
    )
    draft_result = await Runner.run(draft_agent, draft_input, max_turns=DRAFT_MAX_TURNS)
    out: PlannerOutput = draft_result.final_output

    # Guard 1 — real research calls
    n_research = _count_research_calls(draft_result)
    if n_research == 0:
        print("[guard] WARNING: 0 research calls (tool_choice='required' should prevent this)")

    # Guard 2 — URL validator: strip fabricated source_urls
    real_urls = _extract_research_urls(draft_result)
    for theme in out.themes:
        for pt in theme.points:
            if pt.source_url and pt.source_url.rstrip(".,") not in real_urls:
                print(f"[guard] WARNING: fabricated source_url stripped: {pt.source_url}")
                print(f"        (angle: {pt.angle[:70]})")
                pt.source_url = None

    # Guard 3 — first-principles carry-over floor
    fp_points = [pt for t in out.themes for pt in t.points if pt.source_url is None]
    if len(fp_points) < 2:
        print(f"[guard] WARNING: only {len(fp_points)} first-principles points survived")

    return bs, out, n_research


print("run_planner_for_subject + guards defined.")

run_planner_for_subject + guards defined.


In [ ]:
# Cell 8 - run_prep: the end-to-end fan-out
# utterance -> resolve_meeting_context (resolver) -> route -> manager -> N parallel
# brainstorm->draft chains. A semaphore caps simultaneous subjects so a rich profile
# does not hammer scrape targets / LLM concurrency (Brave is already globally rate-limited).

PREP_MAX_CONCURRENCY = 4


async def _prep_fanout(utterance: str, contact_info: str | None):
    """manager -> parallel brainstorm->draft. Returns (ManagerOutput, cards).
    Empty subjects (cold start) -> prints the context-elicitation prompt, returns []."""
    mo = await run_manager(utterance, contact_info)
    print(f"[prep] manager: {len(mo.subjects)} subjects, relationship={mo.relationship}")
    if not mo.subjects:
        print("[prep] no subjects -> need more context: what do you know about them "
              "(interests, how you know them, where you're meeting)?")
        return mo, []

    sem = asyncio.Semaphore(PREP_MAX_CONCURRENCY)

    async def _one(s: Subject):
        async with sem:
            bs, out, n = await run_planner_for_subject(
                s.name, s.phrase, mo.relationship
            )
            return out, n

    results = await asyncio.gather(*[_one(s) for s in mo.subjects])
    cards = [(s, out, n) for s, (out, n) in zip(mo.subjects, results)]
    return mo, cards


async def run_prep(utterance: str, user_id: str = "u1",
                   resolver_mode: Literal["lexical", "hybrid"] = "hybrid"):
    """utterance -> resolve (lexical or hybrid) -> route -> fan-out.
    Returns (ManagerOutput | None, cards, ResolutionResult). On an ambiguous result it
    prints the choices and returns (None, [], resolution); pass that resolution to
    continue_prep(utterance, resolution, choice) once the user picks a candidate."""
    if resolver_mode == "hybrid":
        resolution = await resolve_meeting_context_hybrid(user_id, utterance)
    else:
        resolution = await resolve_meeting_context(user_id, utterance)
    route = route_resolution(resolution, user_id)
    print(f"[prep] resolver({resolver_mode}): {resolution.status} -> {route.action}")
    if route.action == "disambiguate":
        print(route.message)
        return None, [], resolution
    mo, cards = await _prep_fanout(utterance, route.contact_info)
    return mo, cards, resolution


async def continue_prep(utterance: str, resolution, choice, user_id: str = "u1"):
    """Step 6 continuation: user picks from an ambiguous resolution, then prep runs."""
    resolved = select_candidate(resolution, choice)
    route = route_resolution(resolved, user_id)
    print(f"[prep] choice -> {resolved.status} -> {route.action}")
    if route.action != "prep":
        print("[prep] not a valid pick; still ambiguous")
        return None, []
    return await _prep_fanout(utterance, route.contact_info)


def show_prep(mo, cards, elapsed):
    print(f"=== PREP COMPLETE in {elapsed:.1f}s ===")
    print(f"relationship:    {mo.relationship}")
    print(f"overall_context: {mo.overall_context}")
    print(f"subjects:        {[s.name for s in mo.subjects]}")
    print()
    for s, out, n in cards:
        total = sum(len(t.points) for t in out.themes)
        sourced = sum(1 for t in out.themes for pt in t.points if pt.source_url)
        print(f"--- {s.name}  (research={n}, themes={len(out.themes)}, points={total}, sourced={sourced}) ---")
        print(f"  framing: {out.framing}")
        for t in out.themes:
            print(f"  [{t.name}]")
            for pt in t.points:
                tag = "  [src]" if pt.source_url else ""
                print(f"    - {pt.angle}{tag}")
        print()


print("run_prep + continue_prep + show_prep defined.")

In [10]:
# Cell 14 — Full pipeline: utterance -> manager -> fan-out -> synthesize
async def run_full(utterance: str):
    """utterance -> (ManagerOutput, SocialPrepResult). result is None if no subjects."""
    mo, cards, _res = await run_prep(utterance)
    if not cards:
        return mo, None
    subjects = [
        {"subject": s.name, "framing": out.framing,
         "points": [pt.model_dump() for th in out.themes for pt in th.points]}
        for (s, out, n) in cards
    ]
    result = await synthesize(mo.overall_context, subjects)
    return mo, result

print("run_full defined.")


run_full defined.


In [ ]:
# Step 11 - front door: classify intent, then dispatch to the right flow.
async def handle(user_id: str, utterance: str,
                 resolver_mode: Literal["lexical", "hybrid"] = "hybrid"):
    """Front door: classify intent, then dispatch. resolver_mode picks lexical vs
    hybrid (lexical + semantic-memory fallback) resolution for prep and recall.
      prep   -> (ManagerOutput | None, cards, ResolutionResult)  [ambiguous -> continue_prep]
      recall -> RecallResult  (answer / disambiguate / not_found)
      update -> UpdateResult  (proposes; call run_update(..., confirm=True) to commit)"""
    intent = await classify_intent(utterance)
    print(f"[handle] intent={intent}")
    if intent == "recall":
        return await (run_recall_hybrid(user_id, utterance) if resolver_mode == "hybrid"
                      else run_recall(user_id, utterance))
    if intent == "update":
        return await run_update(user_id, utterance)
    return await run_prep(utterance, user_id, resolver_mode)


print("handle (intent dispatch) defined.")

In [ ]:
RUN_DEMO = globals().get("RUN_DEMO", True)
if RUN_DEMO:
    mo, result = await run_full("I am meeting a friend who plays clash royale, and is really into workout lately")
    print("=" * 72)
    print(f"overall_context: {result.overall_context}")
    for subj in result.subjects:
        print(f"\n### {subj.subject}")
        print(f"  framing: {subj.framing}")
        print(f"  reasoning: {subj.reasoning}")
        print(f"  TALKING POINTS ({len(subj.talking_points)}):")
        for pt in subj.talking_points:
            print(f"    - {pt.angle}")
            if pt.context:
                print(f"        context: {pt.context}")
        print(f"  NEWS ({len(subj.news)}):")
        for pt in subj.news:
            print(f"    - {pt.angle}")
            if pt.context:
                print(f"        context: {pt.context}")
            print(f"        source:  {pt.source_url}")
        print(f"  RESERVE ({len(subj.reserve)}): {[pt.angle for pt in subj.reserve]}")
    guard(result)
